In [1]:
import os
import json
import pickle
import uuid
import tempfile


import torch
import soundfile as sf
import gradio as gr


from dotenv import load_dotenv


from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq


from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory


from kokoro import KPipeline

d:\Gitlab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Hybrid RAG Configuration and Vector Store

In [2]:
with open(
    "../hybrid_config.json",
    "r"
) as f:

    hybrid_config = json.load(f)

bge_embeddings = HuggingFaceEmbeddings(

    model_name=hybrid_config["embedding_model"],

    model_kwargs={
        "device": "cpu"
    },

    encode_kwargs={
        "normalize_embeddings": True
    }
)


bge_vector_store = Chroma(

    collection_name="final_rag_collection",

    persist_directory="../chroma_db",

    embedding_function=bge_embeddings
)

K = hybrid_config["K"]


bge_retriever = bge_vector_store.as_retriever(

    search_kwargs={
        "k": K
    }
)

with open(
    "../bm25_retriever.pkl",
    "rb"
) as f:

    bm25_retriever = pickle.load(f)


print("BM25 Retriever loaded.")

class HybridRRF:


    def __init__(
        self,
        retrievers,
        weights=None,
        k=60,
        top_k=3
    ):

        self.retrievers = retrievers

        self.weights = weights

        self.k = k

        self.top_k = top_k



    def invoke(
        self,
        query
    ):

        all_results = []


        for retriever, weight in zip(
            self.retrievers,
            self.weights
        ):


            docs = retriever.invoke(query)


            for rank, doc in enumerate(
                docs,
                start=1
            ):


                score = weight * (
                    1 / (self.k + rank)
                )


                all_results.append({

                    "doc": doc,

                    "score": score

                })



        fused_scores = {}



        for item in all_results:


            content = item["doc"].page_content



            if content not in fused_scores:


                fused_scores[content] = {

                    "doc": item["doc"],

                    "score": 0

                }



            fused_scores[content]["score"] += item["score"]




        ranked_docs = sorted(

            fused_scores.values(),

            key=lambda x: x["score"],

            reverse=True

        )



        return [

            item["doc"]

            for item in ranked_docs[:self.top_k]

        ]


hybrid_retriever = HybridRRF(

    retrievers=[

        bge_retriever,

        bm25_retriever

    ],

    weights=[

        hybrid_config["weights"]["BGE"],

        hybrid_config["weights"]["BM25"]

    ],

    k=hybrid_config["rrf_k"],

    top_k=K

)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12480.06it/s]


BM25 Retriever loaded.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9108\2686495998.py:46: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  bm25_retriever = pickle.load(f)


## Setup LLM for RAG Pipeline

In [3]:
load_dotenv()


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    streaming=True,
    api_key=os.getenv("GROQ_API_KEY")
)

In [4]:

rag_prompt = ChatPromptTemplate.from_template(
"""
You are a helpful assistant.

Answer the question based only on the context below.

If the answer is not available in the context, say:
I don't know.

Context:
{context}

Question:
{question}

Answer:
"""
)


In [5]:
def format_docs(docs):

    return "\n\n".join(
        doc.page_content
        for doc in docs
    )


def rag_stream(question):

    docs = hybrid_retriever.invoke(
        question
    )


    context = format_docs(
        docs
    )


    messages = rag_prompt.invoke(
        {
            "context": context,
            "question": question
        }
    )

    for chunk in llm.stream(messages):

        yield chunk.content

<!-- ## Memory -->

## Add Chat Memory to RAG Pipeline

In [6]:
store = {}


def get_session_history(session_id):

    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()

    return store[session_id]


memory_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are a helpful assistant.

Answer only using the context.

Context:
{context}
"""
        ),

        MessagesPlaceholder(
            variable_name="history"
        ),

        (
            "human",
            "{question}"
        )
    ]
)


def retrieve_context(inputs):

    question = inputs["question"]

    docs = hybrid_retriever.invoke(
        question
    )

    context = format_docs(docs)

    return {
        "context": context,
        "question": question,
        "history": inputs.get("history", [])
    }


rag_chain = (
    retrieve_context
    |
    memory_prompt
    |
    llm
)


rag_chatbot = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history"
)

d:\Gitlab\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:

os.environ["NO_PROXY"] = "localhost,127.0.0.1"
os.environ["no_proxy"] = "localhost,127.0.0.1"

## Text-to-Speech Generation

In [8]:

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)


tts_pipeline = KPipeline(
    lang_code="a"
)


print("Kokoro TTS loaded")



def text_to_speech(text):

    if text is None or text.strip() == "":
        return None



    print("TTS INPUT:")
    print(text)



    output_file = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ).name



    audio_chunks = []



    generator = tts_pipeline(

        text,

        voice="af_heart",

        speed=1.0

    )



    for _, _, audio in generator:

        audio_chunks.append(audio)



    final_audio = torch.cat(
        audio_chunks
    )



    sf.write(

        output_file,

        final_audio.numpy(),

        24000

    )



    return output_file



Using device: cpu


d:\Gitlab\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
d:\Gitlab\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Kokoro TTS loaded


## SESSION MANAGEMENT

In [9]:
sessions = {}

def get_session_id():

    session_id = str(uuid.uuid4())

    sessions[session_id] = True

    return session_id

## CHAT RESPONSE 


In [10]:
def chat_response(

    message,

    history,

    session_id

):


    if history is None:

        history=[]



    if session_id is None:

        session_id=get_session_id()




    if message is None or message.strip()=="":

        return history, session_id, "", ""



    response = rag_chatbot.invoke(

        {
            "question":message
        },

        config={

            "configurable":

            {

                "session_id":session_id

            }

        }

    )



    answer = response.content



    history.append(

        {

            "role":"user",

            "content":message

        }

    )


    history.append(

        {

            "role":"assistant",

            "content":answer

        }

    )





    return (

        history,

        session_id,

        "",

        answer

    )



def clear_chat(session_id):


    try:

        if session_id in store:

            del store[session_id]


    except:

        pass



    new_session = get_session_id()



    return (

        [],

        new_session,

        "",

        "",

        None

    )


## Gradio UI

In [15]:
import gradio as gr


custom_css = """

body {
    background-color: #f5f7fb;
}


#title {
    text-align:center;
    font-size:32px;
    font-weight:bold;
}


#subtitle {
    text-align:center;
    color:#555;
}


.gr-button {
    border-radius:10px !important;
}


#send_btn {
    background:#2563eb;
    color:white;
}


#new_btn {
    background:#dc2626;
    color:white;
}


#voice_btn {
    background:#16a34a;
    color:white;
}

"""


with gr.Blocks(
    css=custom_css,
    theme=gr.themes.Soft()
) as demo:


    gr.HTML(
        """
        <div id="title">
        🏢 HR Knowledge Assistant
        </div>

        """
    )


    chatbot = gr.Chatbot(
        height=550,
        show_label=False
    )


    session_state = gr.State(None)

    answer_state = gr.State("")



    with gr.Row():


        message_box = gr.Textbox(

            placeholder=
            "Ask about employee policies, benefits, holidays...",

            label="",

            scale=5

        )


        send_button = gr.Button(

            "📨 Send",

            elem_id="send_btn",

            scale=1

        )



    with gr.Row():


        speak_button = gr.Button(

            "🔊 Read Answer",

            elem_id="voice_btn"

        )


        new_chat = gr.Button(

            "🗑 New Conversation",

            elem_id="new_btn"

        )



    voice_output = gr.Audio(

        label="🔊 Assistant Voice",

        autoplay=False

    )



    send_button.click(
        chat_response,
        inputs=[
            message_box,
            chatbot,
            session_state
        ],
        outputs=[
            chatbot,
            session_state,
            message_box,
            answer_state
        ]
    )


    message_box.submit(
        chat_response,
        inputs=[
            message_box,
            chatbot,
            session_state
        ],
        outputs=[
            chatbot,
            session_state,
            message_box,
            answer_state
        ]
    )


    speak_button.click(
        text_to_speech,
        inputs=[
            answer_state
        ],
        outputs=[
            voice_output
        ]
    )


    new_chat.click(
        clear_chat,
        inputs=[
            session_state
        ],
        outputs=[
            chatbot,
            session_state,
            message_box,
            answer_state,
            voice_output
        ]
    )



    

demo.launch(
    server_name="127.0.0.1",
    server_port=7848,
    share=False
)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9108\1673927662.py:49: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7848
* To create a public link, set `share=True` in `launch()`.


TTS INPUT:
Hello. Is there something I can help you with regarding the provided context about teamwork and professional development?


d:\Gitlab\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENOVO\.cache\huggingface\hub\models--hexgrad--Kokoro-82M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Exception in callback _ProactorBasePipeTransport._call_connection_lost(None)
handle: <Handle _ProactorBasePipeTransport._call_connect

TTS INPUT:
Hello. Is there something I can help you with regarding the provided context about teamwork and professional development?


In [16]:
demo.close()

Closing server running on port: 7848
